# 🚀 AdamV: Standalone Run
This notebook tests the real-world performance of **AdamV** only, to save time.


In [ ]:
!pip install ninja matplotlib torchvision


In [ ]:

import os
import torch
from torch.utils.cpp_extension import load_inline

# Configure CUDA for T4 GPUs on Kaggle
os.environ['TORCH_CUDA_ARCH_LIST'] = "7.5"
os.environ['NVIDIA_VISIBLE_DEVICES'] = "all"
os.environ['OMP_NUM_THREADS'] = "1"

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

# Include CUDAContext to fix stream issues
cpp_source = """
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>

void adamv_step_cuda(at::Tensor p, at::Tensor grad, at::Tensor exp_avg, at::Tensor exp_avg_sq, at::Tensor direcao, float lr, float beta1, float beta2, float eps, float weight_decay, float progresso, float bakh_thresh_eff, int step, int D, bool omni_triggered, int64_t punning_mask);
"""

cuda_source = """
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <cmath>
#include <ATen/cuda/CUDAContext.h>

#ifndef M_PI
#define M_PI 3.14159265358979323846
#endif

const int BLOCK_SIZE = 256;

template <typename scalar_t>
__global__ void adamv_prepare_kernel(
    float* __restrict__ exp_avg,
    float* __restrict__ exp_avg_sq,
    float* __restrict__ direcao_buffer,
    const scalar_t* __restrict__ grad,
    float beta1, float beta2, float bias_correction1, float bias_correction2, float eps, int numel) {
    
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < numel) {
        float g = static_cast<float>(grad[idx]);
        float m = exp_avg[idx];
        float v = exp_avg_sq[idx];

        float snr = (m * m) / (v + eps);
        float beta1_eff = beta1 + 0.05f * (1.0f - 2.0f * snr);
        beta1_eff = max(0.0f, min(1.0f, beta1_eff));
        
        m = beta1_eff * m + (1.0f - beta1_eff) * g;
        v = beta2 * v + (1.0f - beta2) * g * g;
        
        exp_avg[idx] = m;
        exp_avg_sq[idx] = v;

        float m_hat = m / bias_correction1;
        float v_hat = v / bias_correction2;
        
        direcao_buffer[idx] = m_hat / (sqrt(v_hat) + eps);
    }
}

template <typename scalar_t>
__global__ void adamv_update_kernel(
    scalar_t* __restrict__ params,
    const scalar_t* __restrict__ grad,
    const float* __restrict__ exp_avg_sq,
    const float* __restrict__ direcao_buffer,
    const float* __restrict__ norm_tensor_ptr,
    float progresso, float cooling_factor, float bakh_thresh_eff, float bias_correction2, float eps, 
    float wd_factor, float lr_max, float weight_decay, int numel, int D,
    bool omni_triggered, uint32_t punning_mask) {
    
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < numel) {
        scalar_t p = params[idx];
        
        // MACRO-BRANCH: Uniforme no warp, nÃ£o gera divergence severa
        if (omni_triggered && sizeof(scalar_t) == 4) {
            float p_f = static_cast<float>(p);
            // REGRA: Checagem de robustez bitwise (branchless)
            uint32_t p_int = __float_as_uint(p_f);
            uint32_t abs_int = p_int & 0x7FFFFFFF;
            uint32_t is_valid = (abs_int < 0x7F800000) && (abs_int > 0);
            uint32_t is_valid_mask = 0 - is_valid;
            uint32_t conditional_mask = (punning_mask & is_valid_mask) | (~is_valid_mask);
            
            uint32_t sign = p_int & 0x80000000;
            uint32_t exp  = p_int & 0x7F800000;
            uint32_t mant = p_int & 0x007FFFFF;
            
            // Aplica a mÃ¡scara APENAS na mantissa se vÃ¡lido
            uint32_t mant_mod = mant & conditional_mask;
            
            p_f = __uint_as_float(sign | exp | mant_mod);
            p = static_cast<scalar_t>(p_f);
        }
        
        float g = static_cast<float>(grad[idx]);
        float dir = direcao_buffer[idx];
        float v = exp_avg_sq[idx];
        
        float norm_dir = (*norm_tensor_ptr) / sqrt(static_cast<float>(D));
        float envelope = (1.0f + progresso) / (progresso + norm_dir + eps);
        float lr_efetivo = lr_max * min(envelope * cooling_factor, 1.5f);
        
        float a = lr_efetivo * dir;
        float v_hat = v / bias_correction2;
        float sqrt_v = sqrt(v_hat);
        
        bool explosao_mask = std::abs(g) > (bakh_thresh_eff * sqrt_v);
        
        // Bakhshali Quartic Brake
        float denom = sqrt_v + std::abs(a) + eps;
        float correction = (a * a) / (2.0f * denom);
        float bakhshali_brake = a - copysignf(1.0f, a) * correction;
        
        float step_size = explosao_mask ? bakhshali_brake : a;
        
        if (weight_decay != 0.0f) {
            p = static_cast<scalar_t>(static_cast<float>(p) * (1.0f - lr_max * weight_decay * wd_factor));
        }
        
        params[idx] = static_cast<scalar_t>(static_cast<float>(p) - step_size);
    }
}

#define CHECK_CUDA(x) TORCH_CHECK(x.device().is_cuda(), #x " must be a CUDA tensor")
#define CHECK_CONTIGUOUS(x) TORCH_CHECK(x.is_contiguous(), #x " must be contiguous")
#define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)

void adamv_step_cuda(
    at::Tensor p,
    at::Tensor grad,
    at::Tensor exp_avg,
    at::Tensor exp_avg_sq,
    at::Tensor direcao,
    float lr,
    float beta1,
    float beta2,
    float eps,
    float weight_decay,
    float progresso,
    float bakh_thresh_eff,
    int step,
    int D,
    bool omni_triggered,
    int64_t punning_mask) 
{
    CHECK_INPUT(p);
    CHECK_INPUT(grad);
    CHECK_INPUT(exp_avg);
    CHECK_INPUT(exp_avg_sq);
    CHECK_INPUT(direcao);

    int numel = p.numel();
    int blocks = (numel + BLOCK_SIZE - 1) / BLOCK_SIZE;

    float bias_correction1 = 1.0f - std::pow(static_cast<float>(beta1), static_cast<float>(step));
    float bias_correction2 = 1.0f - std::pow(static_cast<float>(beta2), static_cast<float>(step));

    cudaStream_t stream = at::cuda::getCurrentCUDAStream();

    AT_DISPATCH_FLOATING_TYPES_AND_HALF(p.scalar_type(), "adamv_prepare", [&] {
        adamv_prepare_kernel<scalar_t><<<blocks, BLOCK_SIZE, 0, stream>>>(
            exp_avg.data_ptr<float>(),
            exp_avg_sq.data_ptr<float>(),
            direcao.data_ptr<float>(),
            grad.data_ptr<scalar_t>(),
            beta1, beta2, bias_correction1, bias_correction2, eps, numel
        );
    });

    // Compute norm asynchronously on GPU
    at::Tensor norm_tensor = at::linalg_norm(direcao);
    
    float cooling_factor = 0.5f * (1.0f + std::cos(M_PI * progresso));
    float wd_factor = 0.5f * (1.0f + std::cos(M_PI * progresso));

    AT_DISPATCH_FLOATING_TYPES_AND_HALF(p.scalar_type(), "adamv_update", [&] {
        adamv_update_kernel<scalar_t><<<blocks, BLOCK_SIZE, 0, stream>>>(
            p.data_ptr<scalar_t>(),
            grad.data_ptr<scalar_t>(),
            exp_avg_sq.data_ptr<float>(),
            direcao.data_ptr<float>(),
            norm_tensor.data_ptr<float>(),
            progresso, cooling_factor, bakh_thresh_eff, bias_correction2, eps, wd_factor, lr, weight_decay, numel, D,
            omni_triggered, static_cast<uint32_t>(punning_mask)
        );
    });
}

"""

print("Compiling AdamV CUDA Kernel (JIT) ...")
adamv_cuda = load_inline(
    name='adamv_cuda',
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=['adamv_step_cuda'],
    with_cuda=True,
    extra_cflags=['-O3', '-fopenmp'],
    extra_cuda_cflags=['-O3', '-use_fast_math', '-lineinfo'],
    verbose=True
)
print("CUDA Kernel loaded successfully!")

class DummyCPU:
    pass
adamv_cpp = DummyCPU()


In [ ]:
import torch
import math

class AdamV(torch.optim.Optimizer):
    """
    AdamV (Adam Vedic) Optimizer - Pure Python Version.
    AdamV 3.1: Harmonic Refactor (In-Place VRAM Opt, OMNI State Fix, Modular Flags)
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, 
                 weight_decay=0.01, total_steps=10000, 
                 bakhshali_threshold=10.0, enable_omni=True,
                 lp_kappa=0.1, lp_omega=10.0, punning_mask=0xFFFFE000,
                 enable_ignition=True, enable_cooling=False, enable_brake=True):
                 
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        total_steps=total_steps, bakhshali_threshold=bakhshali_threshold,
                        enable_omni=enable_omni, lp_kappa=lp_kappa, lp_omega=lp_omega, 
                        punning_mask=punning_mask,
                        enable_ignition=enable_ignition, enable_cooling=enable_cooling, enable_brake=enable_brake)
        super(AdamV, self).__init__(params, defaults)
        
        self.state['omni_global'] = {
            'loss_ema': float('inf'),
            'patience': 0,
            'clock_reset_step': 0,
            'global_step': 0,
        }

    @torch.no_grad()
    def step(self, closure=None, current_loss=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
                
        if current_loss is not None:
            loss = current_loss

        g_state = self.state['omni_global']
        g_state['global_step'] += 1
        current_step = g_state['global_step']
        
        omni_triggered = False
        if loss is not None and len(self.param_groups) > 0 and self.param_groups[0]['enable_omni']:
            loss_val = float(loss) if isinstance(loss, torch.Tensor) else loss
            if g_state['loss_ema'] == float('inf'):
                g_state['loss_ema'] = loss_val
                g_state['patience'] = 0
            else:
                g_state['loss_ema'] = 0.9 * g_state['loss_ema'] + 0.1 * loss_val
                
            is_worse = loss_val > g_state['loss_ema'] * 0.99
            g_state['patience'] = g_state['patience'] + 1 if is_worse else 0
            
            patience_limit = max(500, int(self.param_groups[0]['total_steps'] * 0.05))
            if g_state['patience'] >= patience_limit:
                omni_triggered = True
                g_state['patience'] = 0
                g_state['clock_reset_step'] = current_step
                
        for group in self.param_groups:
            lr_max = group['lr']
            total_steps = group['total_steps']
            
            # Autonomous Ignition
            if group.get('enable_ignition', True):
                ignition = min(1.0, current_step / max(1.0, total_steps * 0.10))
                lr_max = lr_max * ignition
                
            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']
            total_steps = group['total_steps']
            bakh_thresh = group['bakhshali_threshold']
            lp_kappa = group['lp_kappa']
            lp_omega = group['lp_omega']
            punning_mask = group['punning_mask']
            
            internal_step = current_step - g_state['clock_reset_step']
            progresso = min(1.0, internal_step / max(1, total_steps))
            
            LP_Fator = 1.0 + lp_kappa * math.cos(lp_omega * math.log(1.0 + progresso * 10.0))
            bakh_thresh_eff = bakh_thresh * LP_Fator
            
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1
                
                # In-Place Momentum Update
                snr = (exp_avg * exp_avg) / (exp_avg_sq + eps)
                beta1_eff = torch.clamp(beta1 + 0.05 * (1.0 - 2.0 * snr), 0.0, 1.0)
                exp_avg.mul_(beta1_eff).add_(grad.float(), alpha=1.0 - beta1_eff)
                
                exp_avg_sq.mul_(beta2).addcmul_(grad.float(), grad.float(), value=1.0 - beta2)
                
                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']
                
                sqrt_v = exp_avg_sq.sqrt().div_(math.sqrt(bias_correction2))
                direcao = (exp_avg / bias_correction1) / (sqrt_v + eps)
                
                norm_dir_padrao = torch.linalg.norm(direcao) / math.sqrt(p.numel())
                
                if group.get('enable_cooling', False):
                    envelope = (1.0 + progresso) / (progresso + norm_dir_padrao + eps)
                    cooling_factor = 0.5 * (1.0 + math.cos(math.pi * progresso))
                    lr_efetivo = lr_max * torch.clamp(envelope * cooling_factor, max=1.5)
                else:
                    lr_efetivo = lr_max
                
                a = direcao.mul_(lr_efetivo)
                
                explosao_mask = torch.abs(grad) > (bakh_thresh_eff * sqrt_v)
                denom = sqrt_v.add_(torch.abs(a)).add_(eps)
                correction = (a * a).div_(denom.mul_(2.0))
                
                bakhshali_brake = a.clone().sub_(torch.sign(a) * correction)
                
                if group.get('enable_brake', True):
                    step_size = torch.where(explosao_mask, bakhshali_brake, a)
                else:
                    step_size = a
                
                if weight_decay != 0:
                    wd_factor = 0.5 * (1.0 + math.cos(math.pi * progresso))
                    p.mul_(1.0 - lr_max * weight_decay * wd_factor)
                    
                p.sub_(step_size)
                
                if omni_triggered:
                    # Deterministic Mantissa Teleportation (Zero-RAM escape)
                    p_int = p.view(torch.int32)
                    sign_exp = p_int & 0xFF800000
                    mant = p_int & 0x007FFFFF
                    
                    mask_val = int(punning_mask)
                    if mask_val > 0x7FFFFFFF:
                        mask_val -= 0x100000000
                    
                    conditional_mask = mask_val if p.dtype == torch.float32 else -1
                    scrambled_mant = (((mant + 1) * 31337) & 0x007FFFFF) & conditional_mask
                    
                    p_new = (sign_exp | scrambled_mant).view(torch.float32)
                    p.copy_(p_new)
                    
                    # Harmonic State Reset: Do not poison momentum. Flush it gracefully.
                    state['exp_avg'].zero_()
                    state['exp_avg_sq'].mul_(0.1)
                
        return loss

class AdamVCpp(torch.optim.Optimizer):
    """
    AdamV (Adam Vedic) Optimizer - C++ Fused Kernel Version.
    AdamV 3.1: Harmonic Refactor
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, 
                 weight_decay=0.01, total_steps=10000, 
                 bakhshali_threshold=10.0, enable_omni=True,
                 lp_kappa=0.1, lp_omega=10.0, punning_mask=0xFFFFE000,
                 enable_ignition=True, enable_cooling=False, enable_brake=True):
                 
        self.adamv_cpp = globals().get('adamv_cuda', None)
        self.adamv_cuda = globals().get('adamv_cuda', None)
        if self.adamv_cuda is None:
            pass # Fallback to python
            
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
            
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        total_steps=total_steps, bakhshali_threshold=bakhshali_threshold,
                        enable_omni=enable_omni, lp_kappa=lp_kappa, lp_omega=lp_omega, 
                        punning_mask=punning_mask,
                        enable_ignition=enable_ignition, enable_cooling=enable_cooling, enable_brake=enable_brake)
        super(AdamVCpp, self).__init__(params, defaults)
        
        self.state['omni_global'] = {
            'loss_ema': float('inf'),
            'patience': 0,
            'clock_reset_step': 0,
            'global_step': 0,
        }

    @torch.no_grad()
    def step(self, closure=None, current_loss=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        if current_loss is not None:
            loss = current_loss

        g_state = self.state['omni_global']
        g_state['global_step'] += 1
        current_step = g_state['global_step']
        
        omni_triggered = False
        if loss is not None and len(self.param_groups) > 0 and self.param_groups[0]['enable_omni']:
            loss_val = float(loss) if isinstance(loss, torch.Tensor) else loss
            if g_state['loss_ema'] == float('inf'):
                g_state['loss_ema'] = loss_val
                g_state['patience'] = 0
            else:
                g_state['loss_ema'] = 0.9 * g_state['loss_ema'] + 0.1 * loss_val
                
            is_worse = loss_val > g_state['loss_ema'] * 0.99
            g_state['patience'] = g_state['patience'] + 1 if is_worse else 0
            
            patience_limit = max(500, int(self.param_groups[0]['total_steps'] * 0.05))
            if g_state['patience'] >= patience_limit:
                omni_triggered = True
                g_state['patience'] = 0
                g_state['clock_reset_step'] = current_step
                
        for group in self.param_groups:
            lr_max = group['lr']
            total_steps = group['total_steps']
            
            if group.get('enable_ignition', True):
                ignition = min(1.0, current_step / max(1.0, total_steps * 0.10))
                lr_max = lr_max * ignition
                
            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']
            total_steps = group['total_steps']
            bakh_thresh = group['bakhshali_threshold']
            lp_kappa = group['lp_kappa']
            lp_omega = group['lp_omega']
            punning_mask = group['punning_mask']
            
            internal_step = current_step - g_state['clock_reset_step']
            progresso = min(1.0, internal_step / max(1, total_steps))
            
            LP_Fator = 1.0 + lp_kappa * math.cos(lp_omega * math.log(1.0 + progresso * 10.0))
            bakh_thresh_eff = bakh_thresh * LP_Fator
            
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                    state['direcao_buffer'] = torch.empty_like(p, memory_format=torch.preserve_format, dtype=torch.float32)
                
                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                state['step'] += 1
                
                # C++ Fused Kernel Call
                # OMNI logic is pushed to the END inside the C++ Kernel now
                mask_val = int(punning_mask)
                if mask_val > 0x7FFFFFFF:
                    mask_val -= 0x100000000
                    
                if p.is_cpu:
                    self.adamv_cpp.adamv_step_cpu(
                        p, grad, exp_avg, exp_avg_sq, state['direcao_buffer'],
                        lr_max, beta1, beta2, eps, weight_decay,
                        float(progresso), float(bakh_thresh_eff), state['step'], p.numel()
                    )
                elif p.is_cuda and self.adamv_cuda is not None and hasattr(self.adamv_cuda, 'adamv_step_cuda'):
                    self.adamv_cuda.adamv_step_cuda(
                        p, grad, exp_avg, exp_avg_sq, state['direcao_buffer'],
                        lr_max, beta1, beta2, eps, weight_decay,
                        float(progresso), float(bakh_thresh_eff), state['step'], p.numel(),
                        bool(omni_triggered), mask_val
                    )
                else:
                    # Python fallback para GPU - 100% In-Place BRCM
                    bias_correction1 = 1 - beta1 ** state['step']
                    bias_correction2 = 1 - beta2 ** state['step']
                    
                    sqrt_v = exp_avg_sq.sqrt()
                    denom_brcm = sqrt_v.clone().add_(torch.abs(grad.float())).add_(eps)
                    bakh_residual = (grad.float() * grad.float()).div_(denom_brcm.mul_(2.0))
                    curvature_shift = bakh_residual.div_(sqrt_v.add_(eps))
                    
                    beta1_eff = torch.exp(-curvature_shift).mul_(beta1)
                    
                    # Update momentum In-Place
                    exp_avg.mul_(beta1_eff).add_(grad.float() * (1.0 - beta1_eff))
                    exp_avg_sq.mul_(beta2).addcmul_(grad.float(), grad.float(), value=1 - beta2)
                    
                    m_hat = exp_avg / bias_correction1
                    v_hat = exp_avg_sq / bias_correction2
                    direcao = m_hat / (v_hat.sqrt() + eps)
                    
                    norm_dir = torch.linalg.norm(direcao) / math.sqrt(p.numel())
                    if group.get('enable_cooling', False):
                        envelope = (1.0 + progresso) / (progresso + norm_dir + eps)
                        cooling = 0.5 * (1.0 + math.cos(math.pi * progresso))
                        lr_efetivo = lr_max * torch.clamp(envelope * cooling, max=1.5)
                    else:
                        lr_efetivo = lr_max
                    
                    a = direcao.mul_(lr_efetivo)
                    sqrt_v_hat = v_hat.sqrt()
                    
                    explosao_mask = torch.abs(grad) > (bakh_thresh_eff * sqrt_v_hat)
                    
                    denom = sqrt_v_hat.add_(torch.abs(a)).add_(eps)
                    correction = (a * a).div_(denom.mul_(2.0))
                    bakhshali_brake = a.clone().sub_(torch.sign(a) * correction)
                    
                    if group.get('enable_brake', True):
                        step_size = torch.where(explosao_mask, bakhshali_brake, a)
                    else:
                        step_size = a
                        
                    if weight_decay != 0:
                        wd_factor = 0.5 * (1.0 + math.cos(math.pi * progresso))
                        p.mul_(1.0 - lr_max * weight_decay * wd_factor)
                        
                    p.sub_(step_size)
                    
                    if omni_triggered:
                        p_int = p.view(torch.int32)
                        sign_exp = p_int & 0xFF800000
                        mant = p_int & 0x007FFFFF
                        
                        conditional_mask = mask_val if p.dtype == torch.float32 else -1
                        scrambled_mant = (((mant + 1) * 31337) & 0x007FFFFF) & conditional_mask
                        
                        p_new = (sign_exp | scrambled_mant).view(torch.float32)
                        p.copy_(p_new)
                        
                        state['exp_avg'].zero_()
                        state['exp_avg_sq'].mul_(0.1)
                        
        return loss



In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
import urllib.request
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import math

# =============================================================================
# 1. Strict Determinism (Data Expert + Skeptical Critic)
# =============================================================================
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed locked to {seed}")

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def get_deterministic_generator(seed: int):
    g = torch.Generator()
    g.manual_seed(seed)
    return g

# =============================================================================
# 2. Dataset Pipeline (Shakespeare-char)
# =============================================================================
class ShakespeareCharDataset(Dataset):
    def __init__(self, seq_len: int = 256, split='train'):
        self.seq_len = seq_len
        file_path = "tinyshakespeare.txt"
        if not os.path.exists(file_path):
            print("Downloading Tiny Shakespeare...")
            url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
            urllib.request.urlretrieve(url, file_path)
            
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
            
        chars = sorted(list(set(text)))
        self.vocab_size = len(chars)
        stoi = {ch: i for i, ch in enumerate(chars)}
        data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
        
        # 90% train, 10% val
        n = int(0.9 * len(data))
        self.data = data[:n] if split == 'train' else data[n:]

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y

def get_nlp_dataloaders(batch_size=64, seq_len=256, seed=42):
    train_ds = ShakespeareCharDataset(seq_len=seq_len, split='train')
    val_ds = ShakespeareCharDataset(seq_len=seq_len, split='val')
    gen = get_deterministic_generator(seed)
    
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, 
                          num_workers=0, worker_init_fn=seed_worker, generator=gen, drop_last=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)
    return train_dl, val_dl, train_ds.vocab_size

# =============================================================================
# 3. Model Architecture (nanoGPT - PhD Design)
# =============================================================================
class Head(nn.Module):
    def __init__(self, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
        self.ffwd = FeedFoward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class NanoGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=384, n_layer=6, n_head=6, block_size=256, dropout=0.2):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

# =============================================================================
# 4. Decoupled Weight Decay & Training Loop
# =============================================================================
@torch.no_grad()
def estimate_loss(model, eval_iters, val_dl, device):
    model.eval()
    losses = []
    val_iter = iter(val_dl)
    for _ in range(eval_iters):
        try:
            X, Y = next(val_iter)
        except StopIteration:
            break
        X, Y = X.to(device), Y.to(device)
        _, loss = model(X, Y)
        losses.append(loss.item())
    model.train()
    return sum(losses)/len(losses) if len(losses) > 0 else 0.0

def configure_optimizers(model, weight_decay, learning_rate, device_type, is_adamv=False):
    param_dict = {pn: p for pn, p in model.named_parameters()}
    param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
    decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
    nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
    optim_groups = [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': nodecay_params, 'weight_decay': 0.0}
    ]
    if is_adamv:
        return AdamVCpp(optim_groups, lr=learning_rate, betas=(0.9, 0.999), enable_omni=False)
    else:
        return torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.999), eps=1e-8)

def run_transformer_arena():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    batch_size = 64
    block_size = 128
    max_iters = 2000
    eval_interval = 200
    eval_iters = 50
    seeds = [42, 1337, 2026] # Multi-seed rigor
    
    results = {"AdamW": [], "AdamVCpp": []}
    
    for seed in seeds:
        print(f"\n--- Running Seed {seed} ---")
        seed_everything(seed)
        train_dl, val_dl, vocab_size = get_nlp_dataloaders(batch_size, block_size, seed)
        
        for opt_name in ["AdamVCpp"]:
            print(f"Training with {opt_name}...")
            # Re-initialize model to guarantee same starting weights per optimizer
            seed_everything(seed)
            model = NanoGPT(vocab_size, block_size=block_size).to(device)
            
            # Independent Hyperparameters
            # We strictly match 1e-3 for a fair 1-to-1 comparison
            lr = 1e-3 if opt_name == "AdamW" else 1e-3
            wd = 0.1
            optimizer = configure_optimizers(model, wd, lr, device, is_adamv=(opt_name=="AdamVCpp"))
            
            if opt_name == "AdamVCpp":
                for group in optimizer.param_groups:
                    group['total_steps'] = max_iters
            
            # Scheduler ONLY for AdamW
            if opt_name == "AdamW":
                scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=max_iters, pct_start=0.1)
            
            train_iter = iter(train_dl)
            history = []
            
            for iter_num in range(max_iters):
                if iter_num % eval_interval == 0 or iter_num == max_iters - 1:
                    val_loss = estimate_loss(model, eval_iters, val_dl, device)
                    ppl = math.exp(val_loss)
                    print(f"Step {iter_num}: Val Loss {val_loss:.4f}, PPL {ppl:.4f}")
                    history.append((iter_num, val_loss))
                
                try:
                    xb, yb = next(train_iter)
                except StopIteration:
                    train_iter = iter(train_dl)
                    xb, yb = next(train_iter)
                
                xb, yb = xb.to(device), yb.to(device)
                
                logits, loss = model(xb, yb)
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                if opt_name == "AdamW":
                    scheduler.step()
                    
            results[opt_name].append(history)
            
    # Plotting Logic
    plt.figure(figsize=(10, 6))
    for opt_name, histories in results.items():
        steps = [h[0] for h in histories[0]]
        losses = np.array([[h[1] for h in history] for history in histories])
        mean_loss = losses.mean(axis=0)
        std_loss = losses.std(axis=0)
        plt.plot(steps, mean_loss, label=opt_name, marker='o')
        plt.fill_between(steps, mean_loss - std_loss, mean_loss + std_loss, alpha=0.2)
        
    plt.title("Transformer Arena (NanoGPT on Shakespeare-char)")
    plt.xlabel("Steps")
    plt.ylabel("Validation Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig("transformer_arena.png")
    print("Transformer Arena Benchmark complete! Saved to assets/transformer_arena.png")

if __name__ == "__main__":
    run_transformer_arena()


In [ ]:
run_transformer_arena()